# DeepWeeds — Phase C: ablation chính trên ConvNeXt-T (C00-C12) + T09 pilot

In [ ]:
import os, sys, platform
REPO_URL = "https://github.com/hdphuc127/K4-Track4-Day2-Deeplearning-Advance.git"
SUB = "submissions/2A202602796-HoDangPhuc/code"
ROOT = "/kaggle/working"
os.chdir(ROOT)
if not os.path.exists("lab"):
    !git clone -q {REPO_URL} lab
else:
    !git -C lab pull -q
CODE = f"{ROOT}/lab/{SUB}"
sys.path.insert(0, CODE)
!pip -q install timm openpyxl
import torch, timm
print("python", platform.python_version(), "| torch", torch.__version__, "| timm", timm.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU")
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import glob, hashlib
os.makedirs("data/labels", exist_ok=True)
# Ảnh: dataset Kaggle ashuraotsuki/deepweeds-fold0 (images.zip nguyên bản Zenodo, MD5 b7b30f96...). Có thể đã được giải nén sẵn.
zips = glob.glob("/kaggle/input/**/images.zip", recursive=True)
if zips:
    h = hashlib.md5()
    with open(zips[0], "rb") as f:
        for c in iter(lambda: f.read(1 << 20), b""): h.update(c)
    assert h.hexdigest() == "b7b30f96d466fba86016aa5a26606e0f", h.hexdigest()
    os.makedirs("data/images", exist_ok=True)
    !unzip -q -n {zips[0]} -d data/images     # zip phẳng: 17.509 file .jpg ở gốc
    IMAGES_DIR = f"{ROOT}/data/images"
else:
    jp = glob.glob("/kaggle/input/**/*.jpg", recursive=True)
    assert jp, "không thấy ảnh trong /kaggle/input: gắn dataset ashuraotsuki/deepweeds-fold0"
    IMAGES_DIR = os.path.dirname(jp[0])
print("IMAGES_DIR", IMAGES_DIR, len(os.listdir(IMAGES_DIR)))
B_ = "https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels"   # nhãn/fold nguyên bản của tác giả
for n in ["labels", "train_subset0", "val_subset0", "test_subset0"]:
    !wget -q -O data/labels/{n}.csv {B_}/{n}.csv
!wc -l data/labels/*.csv

In [ ]:
LABELS_DIR = f"{ROOT}/data/labels"
OUT = f"{ROOT}/out"
os.makedirs(OUT, exist_ok=True)
import experiments as E, train as Tr, dataset as D
import pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
PATHS = dict(images_dir=IMAGES_DIR, labels_dir=LABELS_DIR, out_dir=f"{OUT}/runs",
             pred_dir=f"{OUT}/predictions", curves_dir=f"{OUT}/curves", num_workers=4)
EPOCHS = 12
def cfg(**kw): return Tr.Config(**{**PATHS, "epochs": EPOCHS, **kw})

In [ ]:
BEST = "convnext_tiny"   # chọn sau Bước 1: macro-F1 val 0.971 so với ResNet-50 0.812 (xem báo cáo)
T = {  # exp_id: (trục, override so với C00)
 "C00": ("nền", {}),
 "C01": ("A khởi tạo", dict(init="frozen")),
 "C02": ("A khởi tạo", dict(init="scratch")),
 "C03": ("B augmentation", dict(aug="color")),
 "C04": ("B augmentation", dict(aug="trivial")),
 "C05": ("B augmentation", dict(mix="cutmix")),
 "C06": ("B augmentation", dict(mix="mixup")),
 "C07": ("C loss", dict(loss="ls", label_smoothing=0.1)),
 "C08": ("C loss", dict(loss="focal", focal_gamma=2.0)),
 "C09": ("C loss", dict(loss="ce_weighted", class_weight_beta=0.0)),
 "C10": ("D sampler", dict(sampler="balanced")),
 "C11": ("F EMA", dict(ema_decay=0.995)),
 "C12": ("E LR", dict(lr_backbone=2e-4, lr_head=2e-3)),
}
for k, (ax_, o) in T.items():
    try:
        E.run_resume(cfg(exp_id=k, backbone=BEST, seed=0, **o))
    except Exception as e:
        print("FAILED", k, repr(e)); torch.cuda.empty_cache(); open(f"{OUT}/failed.txt", "a").write(f"{k}: {e!r}
")
# chạy lại T09 của pilot ResNet-50 (lỗi dtype ở phase A đã sửa)
E.run_resume(cfg(exp_id="T09", backbone="resnet50", seed=0, loss="ce_weighted", class_weight_beta=0.0))
Cdf = E.collect_runs(PATHS["out_dir"], "C")
Cdf["delta_vs_C00"] = Cdf.val_macro_f1 - Cdf.loc[Cdf.exp_id == "C00", "val_macro_f1"].iloc[0]
Cdf["axis"] = Cdf.exp_id.map(lambda k: T[k][0]); Cdf.to_csv(f"{OUT}/training_c.csv", index=False)
print(Cdf[["exp_id", "axis", "val_macro_f1", "val_top1", "delta_vs_C00", "train_s_per_epoch"]])

In [ ]:
# bỏ checkpoint để output nhẹ (giữ logits, history, predictions, curves)
!find {OUT}/runs -name "best.pt" -delete
!du -sh {OUT}; ls {OUT}